# Matcher Basics

This notebook mirrors a few simple `Matcher` examples from the Ocelot matcher documentation and keeps them fully self-contained.

The examples here are chosen from the cases that already work in the current codebase and are covered by matcher regression tests.

The next step will be to reuse the same toy lattices for the autodiff workflow.


## Setup

The cell below makes sure the local checkout is importable even if the notebook kernel starts from `demos/ipython_tutorials`.


In [1]:
from pathlib import Path
import sys
import warnings

cwd = Path.cwd().resolve()
for candidate in [cwd, *cwd.parents]:
    if (candidate / 'ocelot').exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break

warnings.filterwarnings(
    'ignore',
    message='Cavity does not declare support for TransferMap; global lattice request falls back to default CavityTM.'
)

import numpy as np
from ocelot import MagneticLattice
from ocelot.cpbd.beam import Twiss
from ocelot.cpbd.elements import Cavity, Drift, Marker, Quadrupole
from ocelot.cpbd.matcher import MatchProblem


initializing ocelot...


In [2]:
def twiss_seed():
    tw0 = Twiss()
    tw0.beta_x = 10.0
    tw0.beta_y = 10.0
    tw0.alpha_x = 0.0
    tw0.alpha_y = 0.0
    tw0.E = 1.0
    return tw0


## 1. Match a Drift Length With a Twiss Target

This is the smallest useful matcher problem: vary one drift length and force the end marker to reach a target `s`.


In [3]:
start = Marker(eid='START')
dvar = Drift(l=0.7, eid='DVAR')
end = Marker(eid='END')
lat = MagneticLattice((start, dvar, end))

problem = MatchProblem(lat, twiss_seed())
problem.vary_element(dvar, 'l', limits=(0.0, None), name='L_DVAR')
problem.target_twiss(end, 's', value=2.2, weight=1.0e6, tol=0.0)

result = problem.solve(solver='ls_trf', max_iter=80, tol=1.0e-10)
print('success =', result.success)
print('matched drift length =', dvar.l)
print('variables =', result.variables)


success = True
matched drift length = 2.2
variables = {'L_DVAR': 2.2}


## 2. Match the End Energy by Varying Cavity Voltage

This mirrors the simple cavity example from the matcher tests. The expected matched voltage is about `0.03 GeV`.


In [4]:
start = Marker(eid='START')
cav = Cavity(l=0.5, v=0.02, phi=0.0, freq=1.3e9, eid='CAV')
end = Marker(eid='END')
lat = MagneticLattice((start, cav, end))

problem = MatchProblem(lat, twiss_seed())
problem.vary_element(cav, 'v', limits=(0.0, 0.05), name='CAV_v')
problem.target_twiss(end, 'E', value=1.03, weight=1.0e6, tol=0.0)

result = problem.solve(solver='ls_trf', max_iter=120, tol=1.0e-12)
print('success =', result.success)
print('matched cavity voltage =', cav.v)
print('variables =', result.variables)


success = True
matched cavity voltage = 0.03000000000000006
variables = {'CAV_v': 0.03000000000000006}


## 3. Target One R-Matrix Element Between Two Internal Markers

This is a direct translation of the documented `target_rmatrix(...)` use case. We force `R12 = 2.7` between `M1` and `M2`.


In [5]:
start = Marker(eid='START')
d0 = Drift(l=0.4, eid='D0')
m1 = Marker(eid='M1')
dvar = Drift(l=1.1, eid='DVAR')
m2 = Marker(eid='M2')
d2 = Drift(l=0.7, eid='D2')
end = Marker(eid='END')
lat = MagneticLattice((start, d0, m1, dvar, m2, d2, end))

problem = MatchProblem(lat, twiss_seed())
problem.vary_element(dvar, 'l', limits=(0.0, 5.0), name='DVAR_l')
problem.target_rmatrix(m1, m2, i=0, j=1, value=2.7, weight=1.0e6)

result = problem.solve(solver='ls_trf', max_iter=100, tol=1.0e-10)
_, r_seg, _ = lat.transfer_maps(energy=1.0, start=m1, stop=m2)
print('success =', result.success)
print('matched drift length =', dvar.l)
print('R12(M1 -> M2) =', r_seg[0, 1])


success = True
matched drift length = 2.7
R12(M1 -> M2) = 2.7


## 4. One Knob Driving Two Quadrupoles

The documentation shows `vary_linked_elements(...)` for one power supply feeding multiple magnets. Here one variable drives `q1.k1` and `q2.k1` with opposite polarity.


In [6]:
start = Marker(eid='S')
q1 = Quadrupole(l=0.2, k1=0.7, eid='Q1')
mid = Drift(l=0.5, eid='D1')
q2 = Quadrupole(l=0.2, k1=-0.7, eid='Q2')
end = Marker(eid='E')
lat = MagneticLattice((start, q1, mid, q2, end))

problem = MatchProblem(lat, twiss_seed())
problem.vary_linked_elements(
    [q1, q2],
    scales=[1.0, -1.0],
    quantity='k1',
    name='PS_Q12',
    limits=(-5.0, 5.0),
)
problem.target_twiss(end, 'beta_x', value=8.0, weight=1.0e6, tol=0.0)

result = problem.solve(solver='ls_trf', max_iter=200, tol=1.0e-10)
print('success =', result.success)
print('power-supply knob =', result.variables['PS_Q12'])
print('q1.k1 =', q1.k1)
print('q2.k1 =', q2.k1)


success = True
power-supply knob = 0.779584856401049
q1.k1 = 0.779584856401049
q2.k1 = -0.779584856401049


## 5. Add a Simple Custom Objective

Not every matcher term needs to be a standard Twiss or `R` target. Here we minimize the final `s` coordinate directly through `minimize_function(...)`.


In [7]:
start = Marker(eid='START')
dvar = Drift(l=1.3, eid='DVAR')
end = Marker(eid='END')
lat = MagneticLattice((start, dvar, end))

problem = MatchProblem(lat, twiss_seed())
problem.vary_element(dvar, 'l', limits=(0.0, 5.0), name='DVAR_l')
problem.minimize_function(lambda state: state.twiss_end.s, name='min_total_s', weight=1.0e4)

result = problem.solve(solver='ls_trf', max_iter=80, tol=1.0e-10)
print('success =', result.success)
print('optimized drift length =', dvar.l)
print('objective reports =', [rep.name for rep in result.objective_reports])


success = True
optimized drift length = 7.748603829682564e-08
objective reports = ['min_total_s']


## Notes on Documentation Examples Not Mirrored Here

A few examples from the documentation are intentionally not copied into this first notebook:

- The `I5` and ring-style examples are meaningful for periodic storage-ring optics, but they are not good first standalone examples for a tutorial notebook.
- The full peak-current optimization example already exists as `demos/ebeam/matcher_ex.py`. It works, but it is much heavier because it tracks particles during every optimizer evaluation.
- Some documentation snippets use placeholders such as `lat`, `tw0`, `q1`, `q2`, `end`, or assume a larger machine lattice. In this notebook I replaced them with minimal self-contained toy lattices so every cell can run on its own.
- The bounds-error example from the docs is intentionally omitted here because it is a failure-mode example by design, not a working usage example.

The next notebook can reuse the same small lattices and replace the matcher evaluation backend with the new autodiff-oriented linear optics path.
